# Traductor RNN - Inferencia Directa (Sin Reentrenar)
## Universidad Autónoma del Caribe (UAC)
### Carga modelo preentrenado desde GitHub y traduce directo en Colab

In [ ]:
# ========================================
# CELDA 1: Instalar dependencias
# ========================================

!pip install torch gradio -q

In [ ]:
# ========================================
# CELDA 2: Clonar repositorio (para tener los modelos)
# ========================================

!rm -rf RNN && git clone https://github.com/VIVA-EL-APRA/RNN.git
%cd RNN
!ls -lh *.pt

In [ ]:
# ========================================
# CELDA 3: Importar librerías
# ========================================

import torch
import torch.nn as nn
import numpy as np

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Dispositivo: {device}')

In [ ]:
# ========================================
# CELDA 4: Vocabulario y arquitectura (idéntica al entrenamiento)
# ========================================

PAD, UNK, SOS, EOS = '<PAD>', '<UNK>', '<SOS>', '<EOS>'

class Vocab:
    def __init__(self):
        self.w2i = {PAD: 0, UNK: 1, SOS: 2, EOS: 3}
        self.i2w = {0: PAD, 1: UNK, 2: SOS, 3: EOS}
        self.n = 4
    def add(self, text):
        for w in text.lower().split():
            if w not in self.w2i:
                self.w2i[w] = self.n
                self.i2w[self.n] = w
                self.n += 1
    def encode(self, text, max_len, sos=False, eos=False):
        ids = []
        if sos: ids.append(self.w2i[SOS])
        for w in text.lower().split():
            ids.append(self.w2i.get(w, self.w2i[UNK]))
        if eos: ids.append(self.w2i[EOS])
        while len(ids) < max_len: ids.append(self.w2i[PAD])
        return ids[:max_len]
    def decode(self, ids):
        ws = []
        for i in ids:
            if torch.is_tensor(i): i = i.item()
            w = self.i2w.get(int(i), UNK)
            if w not in [PAD, SOS, EOS]: ws.append(w)
        return ' '.join(ws)

class Encoder(nn.Module):
    def __init__(self, vs, em, hd, ly, dp):
        super().__init__()
        self.emb = nn.Embedding(vs, em, padding_idx=0)
        self.lstm = nn.LSTM(em, hd, ly, batch_first=True, dropout=dp)
        self.dp = nn.Dropout(dp)
    def forward(self, x):
        e = self.dp(self.emb(x))
        o, (h, c) = self.lstm(e)
        return o, h, c

class Decoder(nn.Module):
    def __init__(self, vs, em, hd, ly, dp):
        super().__init__()
        self.emb = nn.Embedding(vs, em, padding_idx=0)
        self.lstm = nn.LSTM(em, hd, ly, batch_first=True, dropout=dp)
        self.fc = nn.Linear(hd, vs)
        self.dp = nn.Dropout(dp)
    def forward(self, x, h, c):
        e = self.dp(self.emb(x))
        o, (h, c) = self.lstm(e, (h, c))
        return self.fc(o.squeeze(1)), h, c

class Seq2Seq(nn.Module):
    def __init__(self, enc, dec):
        super().__init__()
        self.enc = enc
        self.dec = dec
    def forward(self, src, tgt, tf=0.5):
        bs = src.shape[0]
        max_len = tgt.shape[1]
        out = torch.zeros(bs, max_len, self.dec.fc.out_features).to(src.device)
        _, h, c = self.enc(src)
        dec_in = tgt[:, 0]
        for t in range(1, max_len):
            o, h, c = self.dec(dec_in.unsqueeze(1), h, c)
            out[:, t] = o
            top1 = o.argmax(1)
            dec_in = tgt[:, t] if np.random.random() < tf else top1
        return out

print('Clases definidas ✅')

In [ ]:
# ========================================
# CELDA 5: Cargar checkpoint y reconstruir vocabularios
# ========================================

import os

ckpt_path = 'translator_model.pt' if os.path.exists('translator_model.pt') else 'translator.pt'
print(f'Cargando: {ckpt_path}')
ckpt = torch.load(ckpt_path, map_location=device)

# Reconstruir vocabularios
src_v = Vocab()
tgt_v = Vocab()
src_v.w2i = ckpt['src_vocab']
src_v.i2w = ckpt['src_idx2word']
src_v.n = max(src_v.w2i.values()) + 1
tgt_v.w2i = ckpt['tgt_vocab']
tgt_v.i2w = ckpt['tgt_idx2word']
tgt_v.n = max(tgt_v.w2i.values()) + 1

MAX_LEN = ckpt.get('max_len', 25)
BLEU_SAVED = ckpt.get('bleu_score', None)

print(f'Vocab src: {src_v.n}, tgt: {tgt_v.n}, MAX_LEN: {MAX_LEN}')
if BLEU_SAVED is not None:
    print(f'BLEU guardado: {BLEU_SAVED:.2f}')

In [ ]:
# ========================================
# CELDA 6: Cargar pesos del modelo (sin reentrenar)
# ========================================

EMBED_DIM = 256
HIDDEN_DIM = 512
NUM_LAYERS = 2
DROPOUT = 0.3

enc = Encoder(src_v.n, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
dec = Decoder(tgt_v.n, EMBED_DIM, HIDDEN_DIM, NUM_LAYERS, DROPOUT).to(device)
model = Seq2Seq(enc, dec).to(device)

# Cargar pesos
state_dict = ckpt.get('model_state_dict', ckpt.get('m', None))
if state_dict is not None:
    model.load_state_dict(state_dict)
    print('✅ Pesos del modelo cargados')
else:
    print('⚠️ No se encontró state_dict')

model.eval()
params = sum(p.numel() for p in model.parameters())
print(f'Parámetros: {params:,}')

In [ ]:
# ========================================
# CELDA 7: Función de traducción
# ========================================

def translate(text, direction='EN->ES'):
    if not text.strip():
        return ''
    with torch.no_grad():
        if direction == 'ES->EN':
            src_enc = torch.tensor([tgt_v.encode(text, MAX_LEN)]).to(device)
            _, h, c = enc(src_enc)
            dec_in = torch.tensor([src_v.w2i[SOS]]).to(device)
            vocab = src_v
        else:
            src_enc = torch.tensor([src_v.encode(text, MAX_LEN)]).to(device)
            _, h, c = enc(src_enc)
            dec_in = torch.tensor([tgt_v.w2i[SOS]]).to(device)
            vocab = tgt_v
        result = []
        for _ in range(MAX_LEN):
            o, h, c = dec(dec_in.unsqueeze(1), h, c)
            top = o.argmax(1).item()
            if top == vocab.w2i[EOS] or top == vocab.w2i[PAD]:
                break
            result.append(top)
            dec_in = torch.tensor([top]).to(device)
        return vocab.decode(result)

print('Función translate() lista ✅')

In [ ]:
# ========================================
# CELDA 8: Probar traducciones (sin reentrenar)
# ========================================

tests_en_es = [
    'hello',
    'thank you',
    'i am a student',
    'where is the library',
    'the exam is difficult',
    'i study at the university',
]

print('=== Inglés ➡ Español ===')
for t in tests_en_es:
    print(f'{t:35} -> {translate(t, "EN->ES")}')

tests_es_en = [
    'hola',
    'gracias',
    'soy estudiante',
    'donde esta la biblioteca',
    'el examen es dificil',
]

print('\n=== Español ➡ Inglés ===')
for t in tests_es_en:
    print(f'{t:35} -> {translate(t, "ES->EN")}')

In [ ]:
# ========================================
# CELDA 9: Interfaz Gradio (opcional)
# ========================================

import gradio as gr

def gradio_translate(text, direction):
    return translate(text, direction)

with gr.Blocks() as demo:
    gr.Markdown('# Traductor RNN UAC - Inferencia Directa')
    with gr.Row():
        inp = gr.Textbox(label='Texto a traducir')
        direction = gr.Radio(['EN->ES', 'ES->EN'], label='Dirección', value='EN->ES')
    btn = gr.Button('Traducir')
    out = gr.Textbox(label='Traducción')
    btn.click(fn=gradio_translate, inputs=[inp, direction], outputs=out)

print('Interfaz lista. Ejecuta demo.launch() abajo si quieres usarla:')
# demo.launch()